# Decision Boundary Comparison — SVM, Logistic Regression, Bayes GMM

All three models are trained on all labelled pixels and their decision boundaries are visualised
across three 2D projections of RGB space:
- **R vs G** (B held at its mean)
- **R vs B** (G held at its mean)
- **G vs B** (R held at its mean)

Each plot overlays all three model boundaries with the training data points coloured by true class.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.mixture import GaussianMixture

np.random.seed(42)

## 1. Configuration

In [ ]:
PIXEL_VALUES_XLSX = 'Data\Pixels_All.xlsx'  # change path to your local path

# SVM grid search settings
N_FOLDS = 5
SVM_PARAM_GRID = {
    'svm__C':      [0.1, 1, 10, 100],
    'svm__gamma':  ['scale', 'auto', 0.001, 0.01, 0.1],
    'svm__kernel': ['rbf', 'linear'],
}

# Logistic regression settings
LR_C = 1.0

# Bayes GMM settings
PRIOR_INFECTED = 0.5
N_COMPONENTS   = 2

# Plot settings
MESH_RESOLUTION = 200   # grid points per axis (lower = faster, higher = smoother)

## 2. Load Data

In [ ]:
df = pd.read_excel(PIXEL_VALUES_XLSX, sheet_name='Sheet1')
print(f'Shape: {df.shape}')
print(df['Cluster'].value_counts().rename({0: 'Healthy', 1: 'Infected'}))

X = df[['R', 'G', 'B']].values.astype(np.float64)
y = df['Cluster'].values

R_mean = X[:, 0].mean()
G_mean = X[:, 1].mean()
B_mean = X[:, 2].mean()

print(f'\nChannel means — R: {R_mean:.1f}  G: {G_mean:.1f}  B: {B_mean:.1f}')

## 3. Train Models

In [ ]:
# ── SVM — grid search then refit ──────────────────────────────────────────
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=42, probability=True)),
])

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
grid_search = GridSearchCV(svm_pipeline, SVM_PARAM_GRID, cv=cv,
                           scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X, y)

best = grid_search.best_params_
print(f'Best SVM params: {best}  |  CV F1: {grid_search.best_score_:.4f}')

svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(C=best['svm__C'], gamma=best['svm__gamma'],
                   kernel=best['svm__kernel'], probability=True, random_state=42)),
])
svm_model.fit(X, y)

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────────────
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(C=LR_C, max_iter=1000, random_state=42)),
])
lr_model.fit(X, y)
print('Logistic Regression trained.')

## 4. Interactive 3D Decision Boundaries (Plotly)

The SVM and Logistic Regression boundaries are the 2D surfaces in RGB space where
each model's signed score is zero. Rendered as near-opaque isosurfaces with a hard
contour mesh drawn on the surface, so the boundary reads clearly; training pixels
are shown as a faint 3D scatter. Use the buttons to isolate a model, drag to
rotate, scroll to zoom. Also written to `decision_boundaries_3d.html` as a
standalone interactive file.

- **SVM** — signed distance to the separating hyperplane
- **Logistic Regression** — log-odds

Note that Bayes GMM is not included, as the projection distorts the plot and makes it difficult to read

Both models came out linear in the grid search, so each boundary is a plane.

In [ ]:
try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'])
    import plotly.graph_objects as go

import plotly.io as pio
pio.renderers.default = 'notebook'   # use 'browser' to pop out, 'iframe' if inline fails

In [ ]:
# Signed score for each model:  > 0 -> Infected side,  < 0 -> Healthy side,  0 -> boundary
def svm_score(pixels):
    return svm_model.decision_function(pixels)          # signed distance to hyperplane

def lr_score(pixels):
    return lr_model.decision_function(pixels)           # log-odds

score_models = [
    ('SVM',                 svm_score, '#E53935'),
    ('Logistic Regression', lr_score,  '#1E88E5'),
]

In [ ]:
GRID_RES_3D = 45        # points per axis; 45**3 ~ 91k. Drop to ~35 for snappier rotation, raise to 60 for publication.
PAD = 5

def _axis_range(idx):
    lo = max(0.0,   X[:, idx].min() - PAD)
    hi = min(255.0, X[:, idx].max() + PAD)
    return lo, hi

r_lo, r_hi = _axis_range(0)
g_lo, g_hi = _axis_range(1)
b_lo, b_hi = _axis_range(2)

rr, gg, bb = np.meshgrid(
    np.linspace(r_lo, r_hi, GRID_RES_3D),
    np.linspace(g_lo, g_hi, GRID_RES_3D),
    np.linspace(b_lo, b_hi, GRID_RES_3D),
    indexing='ij',
)
grid3d = np.column_stack([rr.ravel(), gg.ravel(), bb.ravel()])
print(f'Grid points: {grid3d.shape[0]:,}')

In [ ]:
SURFACE_OPACITY = 0.85     # 1.0 = fully solid; lower if the front plane hides the back one
EDGE_WIDTH      = 4        # thickness of the hard contour mesh drawn on each surface
POINT_SIZE      = 3
POINT_OPACITY   = 0.45
VIEW_TILT_DEG   = 18       # 0 = perfectly edge-on to the boundary (plane -> line); ~15-25 keeps some depth

fig = go.Figure()

# One isosurface per model at score == 0 (thin symmetric bracket => a single surface at the boundary)
for name, score_fn, color in score_models:
    vals  = score_fn(grid3d)
    scale = np.percentile(np.abs(vals), 95) or 1.0
    eps   = 0.02 * scale
    fig.add_trace(go.Isosurface(
        x=grid3d[:, 0], y=grid3d[:, 1], z=grid3d[:, 2],
        value=vals,
        isomin=-eps, isomax=eps, surface_count=1,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorscale=[[0, color], [1, color]], showscale=False,
        opacity=SURFACE_OPACITY, flatshading=True,
        contour=dict(show=True, color='#111111', width=EDGE_WIDTH),   # hard mesh outline on the surface
        lighting=dict(ambient=0.6, diffuse=0.9, specular=0.15, roughness=0.6),
        name=f'{name} boundary', showlegend=True,
    ))

# Training pixels coloured by true class
for cls, label, col in [(0, 'Healthy', '#1565C0'), (1, 'Infected', '#C62828')]:
    m = y == cls
    fig.add_trace(go.Scatter3d(
        x=X[m, 0], y=X[m, 1], z=X[m, 2], mode='markers',
        marker=dict(size=POINT_SIZE, color=col, opacity=POINT_OPACITY, line=dict(width=0)),
        name=f'{label} (data)',
    ))

# ── Camera pointed edge-on to the SVM boundary ───────────────────────────
# Boundary normal in raw RGB space via a finite-difference gradient of the score.
def _score_grad(score_fn, x0, h=1.0):
    g = np.zeros(3)
    for i in range(3):
        xp = x0.copy(); xp[i] += h
        xm = x0.copy(); xm[i] -= h
        g[i] = (score_fn(xp[None])[0] - score_fn(xm[None])[0]) / (2 * h)
    return g

ranges  = np.array([r_hi - r_lo, g_hi - g_lo, b_hi - b_lo])
n_raw   = _score_grad(svm_score, X.mean(axis=0))
n_cube  = n_raw / ranges                       # into the normalised cube frame (aspectmode='cube')
n_cube /= np.linalg.norm(n_cube)

up0 = np.array([0.0, 0.0, 1.0])
u   = np.cross(n_cube, up0)
if np.linalg.norm(u) < 1e-6:
    u = np.cross(n_cube, np.array([0.0, 1.0, 0.0]))
u /= np.linalg.norm(u)                          # in-plane direction -> edge-on view

t        = np.deg2rad(VIEW_TILT_DEG)
edge_dir = u * np.cos(t) + n_cube * np.sin(t)
edge_dir /= np.linalg.norm(edge_dir)
DIST     = 2.0
edge_eye = dict(x=float(edge_dir[0]*DIST), y=float(edge_dir[1]*DIST), z=float(edge_dir[2]*DIST))
face_eye = dict(x=float(n_cube[0]*DIST),   y=float(n_cube[1]*DIST),   z=float(n_cube[2]*DIST))

# ── Trace-toggle buttons (data always visible) ──────────────────────────
n_models = len(score_models)
_vis = lambda mask: list(mask) + [True, True]
trace_buttons = [dict(label='Both boundaries', method='update',
                      args=[{'visible': _vis([True] * n_models)}])]
for i, (name, _, _) in enumerate(score_models):
    mask = [j == i for j in range(n_models)]
    trace_buttons.append(dict(label=name, method='update', args=[{'visible': _vis(mask)}]))
trace_buttons.append(dict(label='Data only', method='update',
                          args=[{'visible': _vis([False] * n_models)}]))

# ── Preset camera views for screenshots ────────────────────────────────
view_buttons = [
    dict(label='Edge-on to boundary', method='relayout', args=[{'scene.camera.eye': edge_eye}]),
    dict(label='Face-on to boundary', method='relayout', args=[{'scene.camera.eye': face_eye}]),
    dict(label='Along R', method='relayout', args=[{'scene.camera.eye': dict(x=2.2, y=0,   z=0)}]),
    dict(label='Along G', method='relayout', args=[{'scene.camera.eye': dict(x=0,   y=2.2, z=0)}]),
    dict(label='Along B', method='relayout', args=[{'scene.camera.eye': dict(x=0,   y=0,   z=2.2)}]),
    dict(label='Corner',  method='relayout', args=[{'scene.camera.eye': dict(x=1.5, y=1.5, z=1.2)}]),
]

fig.update_layout(
    title='Interactive 3D Decision Boundaries — SVM vs Logistic Regression',
    scene=dict(
        xaxis=dict(title='R', range=[r_lo, r_hi]),
        yaxis=dict(title='G', range=[g_lo, g_hi]),
        zaxis=dict(title='B', range=[b_lo, b_hi]),
        aspectmode='cube',
        camera=dict(eye=edge_eye, projection=dict(type='orthographic')),
    ),
    updatemenus=[
        dict(type='buttons', direction='right', x=0.5, y=1.13,
             xanchor='center', buttons=trace_buttons),
        dict(type='dropdown', direction='down', x=0.0, y=1.13,
             xanchor='left', showactive=True, buttons=view_buttons),
    ],
    legend=dict(itemsizing='constant'),
    width=950, height=800, margin=dict(l=0, r=0, t=90, b=0),
)

fig.write_html('decision_boundaries_3d.html', include_plotlyjs='cdn')

# One-shot high-res PNG (no manual screenshot). Needs: pip install -U kaleido
try:
    fig.write_image('decision_boundaries_3d.png', scale=3, width=1400, height=1000)
    print('Saved decision_boundaries_3d.png')
except Exception as e:
    print('PNG export skipped — run `pip install -U kaleido`, then re-run this cell. Details:', e)

fig.show()
print('Saved to decision_boundaries_3d.html')